# Methodology: Smart Glasses Simulation Using CNNs

## 1. Dataset Overview
- Using COCO 2017 dataset: 80 object classes, real-world images, annotations.
- Only using subset from `train2017` for simplicity.

## 2. Loading and Visualizing Data
- Load annotations from JSON.
- Visualize sample image with bounding boxes and class labels.

## 3. CNN Model Selection
- We will use a pre-trained YOLOv5 (lightweight version).
- Detect objects and simulate "feedback" by printing out object names.


In [2]:
import os
import json
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pycocotools.coco import COCO

# Path setup
data_dir = '/kaggle/input/coco-2017-dataset/coco2017'
train_images = os.path.join(data_dir, 'train2017')
ann_file = os.path.join(data_dir, 'annotations/instances_train2017.json')

# Load COCO annotations
coco = COCO(ann_file)

# Get category names
cats = coco.loadCats(coco.getCatIds())
cat_names = [cat['name'] for cat in cats]
print(f"COCO Categories ({len(cat_names)}):")
print(cat_names)


loading annotations into memory...
Done (t=18.83s)
creating index...
index created!
COCO Categories (80):
['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']


In [ ]:
import random
from matplotlib.patches import Rectangle

# Get all image IDs
image_ids = coco.getImgIds()

# Pick a random image ID
img_id = random.choice(image_ids)
img_info = coco.loadImgs(img_id)[0]

# Load the image
img_path = os.path.join(train_images, img_info['file_name'])
image = Image.open(img_path)

# Load annotations
ann_ids = coco.getAnnIds(imgIds=img_info['id'])
anns = coco.loadAnns(ann_ids)

# Draw bounding boxes
draw = ImageDraw.Draw(image)
for ann in anns:
    bbox = ann['bbox']
    category_id = ann['category_id']
    label = coco.loadCats(category_id)[0]['name']
    draw.rectangle(
        [bbox[0], bbox[1], bbox[0]+bbox[2], bbox[1]+bbox[3]],
        outline='red',
        width=2
    )
    draw.text((bbox[0], bbox[1]), label, fill='red')

# Show the image
plt.figure(figsize=(10, 10))
plt.imshow(image)
plt.axis('off')
plt.title("Sample COCO Image with Bounding Boxes")
plt.show()


In [ ]:
# Clone YOLOv5 repo
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -r requirements.txt


In [ ]:
import torch
from IPython.display import Image as IPImage

# Load pretrained YOLOv5 model (YOLOv5s = small, fast, accurate enough)
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

# Use the same image you loaded before
results = model(img_path)

# Show predictions
results.print()  # class labels and confidence scores
results.show()   # visual output (bounding boxes on image)


In [ ]:
!pip install pyttsx3
!apt-get install -y espeak ffmpeg libespeak1


In [ ]:
!pip install gTTS


In [ ]:
from gtts import gTTS
from IPython.display import Audio

# Extract detected class names
detected_names = results.names
detected_classes = results.pred[0][:, -1].tolist()
labels = [detected_names[int(cls)] for cls in detected_classes]

# Unique object labels
spoken_output = "I see: " + ", ".join(sorted(set(labels)))
print(spoken_output)

# Convert text to speech
tts = gTTS(text=spoken_output, lang='en')
tts.save("detected_objects.mp3")

# Play the audio
Audio("detected_objects.mp3")
